# Ruta 16 | Estados por periodo y distribución de conteos

Edición pública con datos exclusivamente sintéticos. Conserva el procedimiento indicado del EDA existente. Los resultados son ilustrativos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

COLOR = "#176a46"
n = 64
viajes = pd.DataFrame({
    "fecha_operativa": pd.date_range("2030-01-01", periods=n, freq="12h"),
    "categoria_embudo": ["cancelado" if i % 9 == 0 else "finalizado" for i in range(n)],
    "pasajeros": [8 + (7*i) % 38 for i in range(n)],
})

In [ ]:
def viajes_por_(tabla, parametro: str, columna='categoria_embudo'):
    f = tabla['fecha_operativa'].copy()
    llaves = {'anio': f.dt.year, 'mes': f.dt.to_period('M'), 'semana': f.dt.to_period('W'), 'dia': f.dt.dayofweek + 1}
    if parametro not in llaves:
        raise ValueError(f'parametro debe ser uno de {list(llaves)}')
    return tabla.groupby([llaves[parametro].rename(parametro), columna]).size().unstack(fill_value=0)

def describir(s, nombre):
    q = s.quantile([0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    return pd.Series({'n': len(s), 'Media': s.mean(), 'Desv. estándar': s.std(), 'Mínimo': s.min(), 'P5': q[0.05], 'P25': q[0.25], 'Mediana': q[0.5], 'P75': q[0.75], 'P95': q[0.95], 'P99': q[0.99], 'Máximo': s.max(), 'Asimetría': s.skew(), '% ceros': (s == 0).mean() * 100}, name=nombre)

def hist_simple(ax, s, titulo, color):
    corte = s.quantile(0.995)
    ax.hist(s[s <= corte], bins=50, color=color, edgecolor='white', linewidth=0.5)
    ax.axvline(s.median(), color=COLOR, linestyle='--', linewidth=1)
    ax.text(s.median(), ax.get_ylim()[1] * 0.95, f' mediana {s.median():.0f}', va='top', fontsize=9)
    ax.set_title(f'{titulo} (n = {len(s):,}; se omiten {(s > corte).sum()} sobre {corte:.0f})', loc='left', fontsize=10)
    ax.set_xlabel('por viaje')
    ax.set_ylabel('viajes')
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    ax.grid(axis='y', color='#e6e5e1', linewidth=0.8)
    ax.set_axisbelow(True)

In [ ]:
display(viajes_por_(viajes, "semana"))
display(describir(viajes.pasajeros, "Pasajeros ficticios"))
fig, ax = plt.subplots()
hist_simple(ax, viajes.pasajeros, "Ruta 16 · ejemplo sintético", "#8bc8a2")
plt.show()